In [112]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


plt.style.use("seaborn-v0_8")

## Loading the data

In [113]:
apartments = pd.read_csv("../data/sales.csv")
apartments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 822 entries, 0 to 821
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price_numeric  822 non-null    float64
 1   municipality   822 non-null    object 
 2   condition      822 non-null    object 
 3   rooms          822 non-null    float64
 4   square_m2      822 non-null    float64
 5   equipment      822 non-null    object 
 6   level          822 non-null    int64  
 7   heating        822 non-null    object 
 8   price_per_m2   822 non-null    float64
dtypes: float64(4), int64(1), object(4)
memory usage: 57.9+ KB


In [114]:
from sklearn.model_selection import train_test_split, cross_val_score

X = apartments.drop(labels=["price_numeric", "price_per_m2"], axis="columns")
y = apartments["price_numeric"].map(lambda x: np.log(x))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X['municipality'])

---
## Quickly testing out different models

In [115]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline

non_numeric_CT = ColumnTransformer(transformers=[
  ('cat_OHE', OneHotEncoder(handle_unknown='ignore'), ["condition", "municipality", "equipment", "heating"]),
  ('numeric_transformer', "passthrough", ['level', 'rooms'])
], remainder="drop")

numeric_CT = ColumnTransformer(transformers=[
  ('cat_OHE', OneHotEncoder(handle_unknown='ignore'), ["condition", "municipality", "equipment", "heating"]),
  ('numeric_transformer', StandardScaler(), ['level', 'rooms'])
], remainder="drop")

In [116]:
np.abs(y_train - y_train.median()).median()

np.float64(0.2635845208727208)

In [124]:
from sklearn.linear_model import LinearRegression

lin_reg_pipe_line = Pipeline(steps=[
  ('col_transform', numeric_CT),
  ('lin_reg', LinearRegression())
])

eval_score = cross_val_score(lin_reg_pipe_line, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()
lin_reg_pipe_line.fit(X_train, y_train)
test_score = lin_reg_pipe_line.score(X_test, y_test)
print(f"Evaluation score: {eval_score}, Test score: {test_score}")

Evaluation score: -0.21426659329530467, Test score: 0.4479872279648952


In [123]:
from sklearn.ensemble import RandomForestRegressor
rand_forest_ensemble = Pipeline(steps=[
  ('col_transform', non_numeric_CT),
  ('rand_forest', RandomForestRegressor(random_state=42, n_estimators=100))
])

eval_score = cross_val_score(rand_forest_ensemble, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()
rand_forest_ensemble.fit(X_train, y_train)
test_score = rand_forest_ensemble.score(X_train, y_train)

print(f"Evaluation score: {eval_score}, Test score: {test_score}")

Evaluation score: -0.21157495831804168, Test score: 0.9021522676837106


In [119]:
from sklearn.svm import SVR
svm_pipeline = Pipeline([
    ("preprocess", numeric_CT),   # includes StandardScaler for numeric
    ("svm", SVR(kernel="linear"))
])

eval_score = cross_val_score(svm_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

svm_pipeline.fit(X_train, y_train)
test_score = svm_pipeline.score(X_test, y_test)
print(f"Evaluation score: {eval_score}, Test score: {test_score}")

Evaluation score: -0.21472720815116525, Test score: 0.4548562801634012


In [120]:
from sklearn.linear_model import Ridge

ridge_pipeline = Pipeline(steps=[
    ("preprocess", numeric_CT),
    ("ridge", Ridge(alpha=1.0))
])

eval_score = cross_val_score(ridge_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

ridge_pipeline.fit(X_train, y_train)
test_score = ridge_pipeline.score(X_test, y_test)
print(f"Evaluation score: {eval_score}, Test score: {test_score}")

Evaluation score: -0.21398257231866938, Test score: 0.4472491623115039


In [121]:
from sklearn.linear_model import SGDRegressor
sgd_pipeline = Pipeline(steps=[
  ("preprocess", numeric_CT),
  ("SGD", SGDRegressor())
])
cross_val_score(sgd_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

np.float64(-0.28250795612708324)

In [122]:
from sklearn.tree import DecisionTreeRegressor
tree_pipeline = Pipeline(steps=[
  ("preprocess", numeric_CT),
  ("tree", DecisionTreeRegressor())
])
cross_val_score(tree_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error').mean()

np.float64(-0.27031497213870825)

---
# Hyperparameter Tuning / Model Selection / Feature subsetting

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.compose import TransformedTargetRegressor

model_pipeline = Pipeline(steps=[
  ("preprocess", numeric_CT),
  ("model", DecisionTreeRegressor())
])

GridSearchCV